In [3]:
import os
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from next_best_action_env.env import NextBestActionEnv
import numpy as np

In [ ]:

# ================================
# Hàm generate test state
# ================================
def generate_test_state(gender, marital_status, prefix, num_actions=21, 
                        max_queue=5, mean_blood_test_time=10, mean_urine_test_time=5, seed=None):
    rng = np.random.default_rng(seed)
    queue_lengths = rng.integers(low=0, high=max_queue+1, size=num_actions).astype(float)

    actions = [
        "Registration", "Payment", "Get Triage Number", "Measure Vital Signs",
        "General Medicine Examination", "Eye Examination", "ENT Examination",
        "Dental Examination", "Gynecological Examination", "Breast Examination",
        "Blood Test", "Urine Test", "In-depth Eye Examination", "ENT Endoscopy",
        "Electrocardiogram (ECG)", "Post-bronchodilator Spirometry",
        "General Ultrasound", "Cardiac Ultrasound", "Chest X-ray",
        "DEXA Bone Density Scan", "Conclusion"
    ]

    try:
        blood_idx = actions.index("Blood Test")
    except ValueError:
        blood_idx = None
    try:
        urine_idx = actions.index("Urine Test")
    except ValueError:
        urine_idx = None

    blood_result_time = float(rng.uniform(0, mean_blood_test_time)) if blood_idx is not None and prefix[blood_idx]==1 else -1.0
    urine_result_time = float(rng.uniform(0, mean_urine_test_time)) if urine_idx is not None and prefix[urine_idx]==1 else -1.0

    state_list = [gender, marital_status] + prefix + queue_lengths.tolist() + [blood_result_time, urine_result_time]
    return np.array(state_list, dtype=np.float32)

# ================================
# Prefixes theo từng nhóm
# ================================
prefixes_for_male = [
    [0]*21, 
    [1]+[0]*20, 
    [1]*2+[0]*19, 
    [1]*3+[0]*18, 
    [1]*4+[0]*17, 
    [1]*5+[0]*16, 
    [1]*5+[1,0,0]+[0]*13, 
    [1]*5+[1,1,0]+[0]*13,
    [1]*5+[1,1,1]+[0]*13,
    [1]*8+[0,0]+[0]*11,
    [1,1,1,1,1,1,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0],
    [1,1,1,1,1,1,1,1,0,0,1,1,0,1,0,0,0,0,0,0,0],
    [1]*8 + [0]*2 + [1]*10 + [0]
]

prefixes_for_female_married = [
    [0]*21, [1,0]+[0]*19, [1,1,1,1,1]+[0]*16,
    [1]*10 + [0]*11,
    [1,1,1,1,1,1,1,1,1,1,0,1,0,1,0,0,0,0,0,0,0],
    [1,1,1,1,1,1,1,1,1,1,1,1,0,1,0,0,0,0,0,0,0],
    [1]*20 + [0]
]

prefixes_for_female_single = [
    [0]*21, [1,0]+[0]*19, [1,1,1,1,1]+[0]*16,
    [1]*9 + [0] + [0]*11,
    [1,1,1,1,1,1,1,1,0,1,0,1,0,1,0,0,0,0,0,0,0],
    [1,1,1,1,1,1,1,1,0,1,1,1,0,1,0,0,0,0,0,0,0],
    [1]*8 + [0,1] + [1]*10 + [0]
]

# ================================
# Tạo tất cả states test
# ================================
all_states = []
for prefix in prefixes_for_male:
    all_states.append(generate_test_state(gender=1, marital_status=1, prefix=prefix))
for prefix in prefixes_for_male:
    all_states.append(generate_test_state(gender=1, marital_status=0, prefix=prefix))
for prefix in prefixes_for_female_married:
    all_states.append(generate_test_state(gender=0, marital_status=1, prefix=prefix))
for prefix in prefixes_for_female_single:
    all_states.append(generate_test_state(gender=0, marital_status=0, prefix=prefix))

print(f"Generated {len(all_states)} test states")

Generated 28 test states


In [5]:
env = gym.make('NextBestActionEnv-v0')

In [6]:
all_states[0]

array([ 1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  1.,  0.,
        5.,  4.,  0.,  1.,  3.,  4.,  3.,  4.,  1.,  2.,  1.,  4.,  1.,
        3.,  0.,  2.,  4.,  5., -1., -1.], dtype=float32)

In [8]:
for i in range(0,21,1):
    print(env.unwrapped.get_reward(all_states[0], action=i))

(np.float64(9.814814814814815), True, {'action': 'Registration'})
(-10.0, False, {'action': 'Payment', 'violation': 'must_do_before', 'missing': 'Registration'})
(-10.0, False, {'action': 'Get Triage Number', 'violation': 'must_do_before', 'missing': 'Registration'})
(-10.0, False, {'action': 'Measure Vital Signs', 'violation': 'must_do_before', 'missing': 'Registration'})
(-10.0, False, {'action': 'General Medicine Examination', 'violation': 'must_do_before', 'missing': 'Registration'})
(-10.0, False, {'action': 'Eye Examination', 'violation': 'must_do_before', 'missing': 'Registration'})
(-10.0, False, {'action': 'ENT Examination', 'violation': 'must_do_before', 'missing': 'Registration'})
(-10.0, False, {'action': 'Dental Examination', 'violation': 'must_do_before', 'missing': 'Registration'})
(-10.0, False, {'action': 'Gynecological Examination', 'violation': 'must_not_do'})
(-10.0, False, {'action': 'Breast Examination', 'violation': 'must_not_do'})
(-10.0, False, {'action': 'Bloo